In [1]:
from voc_analysis import *
%load_ext autoreload
%autoreload 2

def preprocess_games(sample_games):
    # For cumulative move time, cumulative count per player, vectorize and avoid temp columns
    sample_games = sample_games.sort_values(["gid", "move_ply"], kind="mergesort").reset_index(drop=True)

    mt = sample_games["move_time"].fillna(0).clip(lower=0)
    inc = sample_games["clock_increment"]
    init_clock = sample_games["initial_clock"]

    # Cumulative sum and count grouped by game + color, using assign for functional style
    group_keys = ["gid", "player_white"]
    cum_move_time = mt.groupby([sample_games[k] for k in group_keys]).cumsum()
    num_prior_moves = sample_games.groupby(group_keys, sort=False).cumcount()

    # Time spent before move, then remaining clock (clipped to 0), then after move+increment
    time_spent_prior = cum_move_time - mt
    sample_games["time_left"] = (init_clock - time_spent_prior + num_prior_moves * inc).clip(lower=0)
    sample_games["time_left_after"] = (sample_games["time_left"] - mt + inc).clip(lower=0)
    return sample_games

In [ ]:
# Pick N complete games (metadata filters), then every half-move for those gids.
GAMES_CACHE = "full_games_moves.parquet"
N_GAMES = 500

need_reload = not os.path.exists(GAMES_CACHE)
if os.path.exists(GAMES_CACHE):
    sample_games = pd.read_parquet(GAMES_CACHE)
    need_reload = need_reload or ("clock_increment" not in sample_games.columns)
if need_reload:
    # utils.get_db_connection defaults (20GB, many threads) can OOM small hosts / Jupyter.
    conn = get_db_connection(threads=4, memory_limit="6GB")
    start_date = "2022-01-01"
    end_date = "2022-12-31"
    db_path = '/scratch/gpfs/GRIFFITHS/chess-db/lichess.db'
    try:
        conn.execute(f"ATTACH '{db_path}' AS core (READ_ONLY)")
    except Exception as e:
        print(f"Warning attaching database: {e}")

    sample_games = conn.sql(f"""
        WITH picked AS (
            SELECT g.gid
            FROM core.games g
            WHERE g.utc_datetime BETWEEN '{start_date}' AND '{end_date}'
            AND g.initial_clock >= 300
            AND g.white_elo >= 2000
            AND g.black_elo >= 2000
            ORDER BY g.gid
            LIMIT {N_GAMES}
        )
        SELECT m.gid, m.board_position, m.move_time, m.move_ply, m.player_white,
            g.white_elo, g.black_elo, g.initial_clock, g.clock_increment
        FROM core.moves m
        JOIN core.games g ON m.gid = g.gid
        INNER JOIN picked p ON m.gid = p.gid
        ORDER BY m.gid, m.move_ply
    """).df()
    conn.close()
    sample_games.to_parquet(GAMES_CACHE)

print(f"{sample_games['gid'].nunique()} games, {len(sample_games)} plys")

200 games, 14511 plys


In [3]:
df = preprocess_games(sample_games)

np.int64(0)

: 

In [4]:
EPSILON = 1e-6
df["log_time_left"] = np.log(df["time_left"] + EPSILON)
df["log_move_time"] = np.log(df["move_time"] + EPSILON)

In [5]:
from sklearn.linear_model import LinearRegression

# Prepare the data for regression
X = df[["log_time_left"]].values
y = df["log_move_time"].values

# Fit linear regression model
lr = LinearRegression()
lr.fit(X, y)

print("Linear Regression Results:")
print(f"Intercept: {lr.intercept_}")
print(f"Coefficient for log_time_left: {lr.coef_[0]}")

# Optional: display an R^2 score
print(f"R^2 score: {lr.score(X, y)}")

ValueError: Input y contains NaN.

np.int64(400)